In [0]:
from pyspark.sql.functions import col, sha2, current_timestamp, lit

class ComplianceManager:
    def __init__(self, env):
        self.env = env
        self.critical_keys = ["transaction_id", "customer_id"]

    def validate_and_mask(self, df):
        # 1. Compliance: Mask PII
        df = df.withColumn("masked_email", sha2(col("email"), 256)).drop("email")
        
        # 2. Add Audit Metadata
        df = df.withColumn("processed_at", current_timestamp()).withColumn("origin_env", lit(self.env))
        
        # 3. Split: Valid vs Quarantine (Null Keys)
        is_bad = col("transaction_id").isNull() | col("customer_id").isNull()
        
        valid_df = df.filter(~is_bad)
        quarantine_df = df.filter(is_bad).withColumn("reason", lit("Missing Critical Keys"))
        
        return valid_df, quarantine_df
